# Universo Competitivo — CVM Dados Abertos

Segundo notebook do pipeline. O primeiro (`coleta_fundos_kinea.ipynb`) mapeia os fundos Kinea na prateleira da XP; este monta o **universo competitivo** de cada um e anexa o **patrimônio líquido (PL)** comparável.

**Reprodutibilidade:** roda de cima para baixo, sem estado implícito. Toda a lógica pesada vive em `src/` — este notebook só orquestra e mostra resultados. Só 2 módulos importam: `build_competitive_universe.py` e `patrimonio_liquido.py` (que por sua vez reusa `standardize.py` e `enrich_fii_cnpj.py` — nenhum `.py` órfão fora desse fluxo).

**Fontes CVM (4, cada uma cobrindo um tipo de estrutura regulatória):**

| Fonte | Cobre | Traz PL? |
|---|---|---|
| `registro_fundo_classe.zip` (RCVM175) | Fundos abertos/previdência adaptados | Sim (nível de fundo) |
| `cad_fi.csv` (legado) | Fundos não adaptados | Sim |
| `inf_mensal_fii` (geral + complemento) | FIIs | Sim (no *complemento*, não no *geral*) |
| `inf_mensal_fiagro` | FIAGRO (KNCA11) | Sim |

**Pré-requisito:** ter rodado `coleta_fundos_kinea.ipynb` e `src/ingestion/cvm_download.py` (este lê `data/processed/universo_kinea.csv` e `data/raw/cvm_*`).

**Nota importante:** a ficha bruta da XP não expõe CNPJ de FII — os 7 FIIs Kinea chegam aqui sem CNPJ. A célula 3 enriquece isso explicitamente (usando o mapa já validado em `enrich_fii_cnpj.py`) antes de qualquer join com a CVM.

## 1. Instalar dependências

In [134]:
%pip install pandas requests --quiet

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## 2. Imports e configuração

In [135]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis.build_competitive_universe import montar_universo_competitivo, enriquecer_cnpj_fii
from src.analysis.patrimonio_liquido import montar_universo_kinea_completo

KINEA_UNIVERSO_PATH = PROJECT_ROOT / "data" / "processed" / "universo_kinea.csv"
OUT_RESUMO         = PROJECT_ROOT / "data" / "processed" / "universo_competitivo_resumo.csv"
OUT_CONCORRENTES   = PROJECT_ROOT / "data" / "processed" / "fundos_concorrentes.csv"
OUT_KINEA_COMPLETO = PROJECT_ROOT / "data" / "processed" / "universo_kinea_completo.csv"

print("universo_kinea.csv existe?", KINEA_UNIVERSO_PATH.exists())

universo_kinea.csv existe? True


## 3. Conferir/enriquecer CNPJ dos FIIs (explícito, não escondido)

A ficha da XP não traz CNPJ de FII. `montar_universo_competitivo` já faz esse enriquecimento internamente, mas mostramos aqui pra deixar visível quantos fundos precisaram disso — se esse número mudar (ex: XP passar a mostrar CNPJ), é um sinal de que a lógica pode ser simplificada.

In [136]:
df_kinea = pd.read_csv(KINEA_UNIVERSO_PATH)
print(f"Fundos sem CNPJ na ficha XP: {df_kinea['cnpj'].isna().sum()} de {len(df_kinea)}")

df_kinea_enriquecido = enriquecer_cnpj_fii(df_kinea)
print(f"Depois do enriquecimento: {df_kinea_enriquecido['cnpj'].isna().sum()} sem CNPJ (esperado 0)")

Fundos sem CNPJ na ficha XP: 0 de 17
Depois do enriquecimento: 0 sem CNPJ (esperado 0)


## 4. Montar o universo competitivo

`montar_universo_competitivo` baixa as 4 fontes CVM, enriquece o CNPJ dos FIIs internamente, cruza os fundos Kinea por CNPJ e monta a lista de concorrentes por categoria.

Retorna o resumo (1 linha por fundo Kinea) e os concorrentes (1 linha por par concorrente×fundo-Kinea).

In [137]:
df_resumo, df_concorrentes = montar_universo_competitivo(df_kinea, project_root=PROJECT_ROOT)

n_com_universo = df_resumo["n_universo_inicial"].notna().sum()
print(f"Fundos Kinea com universo calculado: {n_com_universo} de {len(df_resumo)}")
assert n_com_universo == 17, f"Esperado 17, veio {n_com_universo} — investigar antes de prosseguir"
print(f"Total de linhas de concorrente: {len(df_concorrentes):,}")
df_resumo

Fundos Kinea com universo calculado: 17 de 17
Total de linhas de concorrente: 27,938


,fundo_kinea,categoria_xp,classe_anbima_cvm,n_universo_inicial,observacao
0,Kinea Chronos FIM RL - Subclasse I,Multimercado - Macro,Multimercados Macro,648,"Tamanho real da classe (fontes CVM combinadas,..."
1,Kinea Rendimentos Imobiliarios FII (KNCR11),Fundo Imobiliário (FII),FII - Multicategoria,660,"Tamanho real da classe (fontes CVM combinadas,..."
2,Kinea Atlas II FIM RL - Subclasse I,Multimercado - Macro,Multimercados Macro,648,"Tamanho real da classe (fontes CVM combinadas,..."
3,Kinea Oportunidade FIF RF CP RL - Subclasse I,Renda Fixa,Renda Fixa Duração Livre Crédito Livre,3062,"Tamanho real da classe (fontes CVM combinadas,..."
4,Kinea Gama FIF CIC em Acoes RL - Subclasse I,Ações,Ações Livre,1856,"Tamanho real da classe (fontes CVM combinadas,..."
5,Kinea IPCA Dinamico II FIF RF RL - Subclasse II,Renda Fixa,Renda Fixa Duração Livre Crédito Livre,3062,"Tamanho real da classe (fontes CVM combinadas,..."
6,Kinea Alpes Prev XP Seg RF CP FICFI,Multimercado,Previdência Multimercado Livre,2458,"Tamanho real da classe (fontes CVM combinadas,..."
7,Kinea Renda Imobiliaria FII (KNRI11),Fundo Imobiliário (FII),FII - Multicategoria,660,"Tamanho real da classe (fontes CVM combinadas,..."
8,Kinea Indices de Precos FII (KNIP11),Fundo Imobiliário (FII),FII - Multicategoria,660,"Tamanho real da classe (fontes CVM combinadas,..."
9,Kinea Infra FII (KDIF11),Outros,Renda Fixa Duração Livre Crédito Livre,3062,"Tamanho real da classe (fontes CVM combinadas,..."


## 5. Anexar PL comparável dos 17 fundos Kinea

`montar_universo_kinea_completo` extrai o PL de cada fundo Kinea **na mesma fonte/conceito usado para os concorrentes** e calcula o percentil dentro da categoria. Sempre retorna exatamente 17 linhas (garantido por `assert` dentro da função — nunca duplica por causa de CNPJ ausente).

In [138]:
df_kinea_completo = montar_universo_kinea_completo(df_kinea_enriquecido, df_concorrentes)
assert len(df_kinea_completo) == 17, f"Esperado 17 linhas, veio {len(df_kinea_completo)}"

print("Status de PL dos 17 fundos:")
print(df_kinea_completo["status_pl"].value_counts())
df_kinea_completo[["nome_padronizado", "patrimonio_liquido_cvm", "classe_anbima_cvm", "percentil_pl", "pl_sobre_mediana_universo"]]

Status de PL dos 17 fundos:
encontrado    17
Name: status_pl, dtype: int64


,nome_padronizado,patrimonio_liquido_cvm,classe_anbima_cvm,percentil_pl,pl_sobre_mediana_universo
0,Kinea Chronos FIM RL - Subclasse I,8.585835e+07,Multimercados Macro,63.2,1.95
1,Kinea Rendimentos Imobiliarios FII (KNCR11),1.097869e+10,FII - Multicategoria,100.0,126.39
2,Kinea Atlas II FIM RL - Subclasse I,6.423834e+07,Multimercados Macro,57.8,1.46
3,Kinea Oportunidade FIF RF CP RL - Subclasse I,5.380888e+08,Renda Fixa Duração Livre Crédito Livre,88.3,10.55
4,Kinea Gama FIF CIC em Acoes RL - Subclasse I,1.083603e+05,Ações Livre,0.6,0.00
5,Kinea IPCA Dinamico II FIF RF RL - Subclasse II,7.034152e+07,Renda Fixa Duração Livre Crédito Livre,57.5,1.38
6,Kinea Alpes Prev XP Seg RF CP FICFI,6.453379e+09,Previdência Multimercado Livre,99.5,154.46
7,Kinea Renda Imobiliaria FII (KNRI11),4.608085e+09,FII - Multicategoria,99.2,53.05
8,Kinea Indices de Precos FII (KNIP11),7.435552e+09,FII - Multicategoria,99.7,85.60
9,Kinea Infra FII (KDIF11),2.787520e+09,Renda Fixa Duração Livre Crédito Livre,97.2,54.66


## 6. Validação cruzada (ficha XP × CVM)

Para os FIIs que já tinham PL na ficha pública da XP, o PL vindo da CVM deve bater (diferença pequena, ~2% ou menos).

In [139]:
checagem = df_kinea_completo[df_kinea_completo["valor_patrimonial_bruto"].notna()][
    ["nome_padronizado", "valor_patrimonial_bruto", "patrimonio_liquido_cvm"]
]
checagem

,nome_padronizado,valor_patrimonial_bruto,patrimonio_liquido_cvm
1,Kinea Rendimentos Imobiliarios FII (KNCR11),R$ 7.8 bi,1.097869e+10
7,Kinea Renda Imobiliaria FII (KNRI11),R$ 4.6 bi,4.608085e+09
8,Kinea Indices de Precos FII (KNIP11),R$ 7.3 bi,7.435552e+09
10,Kinea Fundo de Fundos Imobiliarios FII (KFOF11),R$ 633 mi,6.196453e+08
12,Kinea High Yield CRI FII (KNHY11),R$ 3 bi,3.059928e+09


## 7. Salvar os resultados

In [140]:
OUT_RESUMO.parent.mkdir(parents=True, exist_ok=True)
df_resumo.to_csv(OUT_RESUMO, index=False, encoding="utf-8")
df_concorrentes.to_csv(OUT_CONCORRENTES, index=False, encoding="utf-8")
df_kinea_completo.to_csv(OUT_KINEA_COMPLETO, index=False, encoding="utf-8")

print("Salvos:")
print(" -", OUT_RESUMO.name)
print(" -", OUT_CONCORRENTES.name, f"({len(df_concorrentes):,} linhas)")
print(" -", OUT_KINEA_COMPLETO.name, f"({len(df_kinea_completo)} fundos)")

Salvos:
 - universo_competitivo_resumo.csv
 - fundos_concorrentes.csv (27,938 linhas)
 - universo_kinea_completo.csv (17 fundos)


## 8. Visão consolidada — 4 dimensões (Produto, Retorno e risco, Conteúdo)

In [141]:
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "universo_kinea_raw.csv"
df_raw = pd.read_csv(RAW_PATH)

# Campos esperados por template de ficha (ver docs/metodologia.md, seção
# "Estrutura de ficha por tipo de produto"). fundo_aberto e previdencia usam
# o mesmo conjunto de rótulos na extração (LABELS_FUNDO_ABERTO) - se algum
# campo faltar de verdade num tipo específico, isso aparece no resultado,
# não é assumido aqui.
TEMPLATE_CAMPOS_ESPERADOS = {
    "fundo_aberto": ["objetivo", "benchmark", "tributacao", "risco_pontuacao_xp", "classificacao_xp", "classificacao_cvm"],
    "previdencia":  ["objetivo", "benchmark", "tributacao", "risco_pontuacao_xp", "classificacao_xp", "classificacao_cvm"],
    "fii":          ["segmento", "dividend_yield", "quantidade_cotistas", "valor_patrimonial"],
}

def calcular_completude(row):
    esperados = TEMPLATE_CAMPOS_ESPERADOS.get(row.get("tipo_pagina"), [])
    if not esperados:
        return pd.Series({"campos_esperados": None, "campos_presentes": None,
                           "completude_conteudo_pct": None, "campos_faltando": None})
    presentes = [c for c in esperados if c in row.index and pd.notna(row[c]) and str(row[c]).strip() != ""]
    faltando = [c for c in esperados if c not in presentes]
    return pd.Series({
        "campos_esperados": len(esperados),
        "campos_presentes": len(presentes),
        "completude_conteudo_pct": round(100 * len(presentes) / len(esperados), 1),
        "campos_faltando": ", ".join(faltando) if faltando else None,
    })

df_completude = pd.concat([df_raw[["url"]], df_raw.apply(calcular_completude, axis=1)], axis=1)

# Junta com Produto + Retorno/risco/porte (df_kinea_completo, já em memória
# da Seção 5) - join por url, nunca por nome (mesma regra do resto do
# pipeline: CNPJ > ticker > nome; aqui url é o identificador estável comum
# aos dois lados, já que df_kinea_completo tem CNPJ enriquecido e df_raw não).
df_visao = df_kinea_completo.merge(df_completude, on="url", how="left")

cols_visao = [
    "nome_padronizado", "tipo_pagina", "categoria_padronizada",
    "taxa_administracao_pct", "percentil_pl", "pl_sobre_mediana_universo",
    "classe_anbima_cvm", "completude_conteudo_pct", "campos_faltando",
]
df_visao_final = df_visao[cols_visao].sort_values("percentil_pl", ascending=False)

OUT_VISAO = PROJECT_ROOT / "data" / "processed" / "visao_consolidada.csv"
df_visao_final.to_csv(OUT_VISAO, index=False, encoding="utf-8")
print("Salvo em:", OUT_VISAO)
df_visao_final

Salvo em: /Users/julianamurakami/Downloads/case-kinea-bi/data/processed/visao_consolidada.csv


,nome_padronizado,tipo_pagina,categoria_padronizada,taxa_administracao_pct,percentil_pl,pl_sobre_mediana_universo,classe_anbima_cvm,completude_conteudo_pct,campos_faltando
11,Kinea Credito Agro FIAGRO FII (KNCA11),fii,Fundo Imobiliário (FII),1.00,100.0,4.62,FIAGRO,50.0,"quantidade_cotistas, valor_patrimonial"
1,Kinea Rendimentos Imobiliarios FII (KNCR11),fii,Fundo Imobiliário (FII),1.00,100.0,126.39,FII - Multicategoria,100.0,NaN
8,Kinea Indices de Precos FII (KNIP11),fii,Fundo Imobiliário (FII),1.00,99.7,85.60,FII - Multicategoria,100.0,NaN
6,Kinea Alpes Prev XP Seg RF CP FICFI,previdencia,Multimercado,1.00,99.5,154.46,Previdência Multimercado Livre,66.7,"objetivo, classificacao_xp"
7,Kinea Renda Imobiliaria FII (KNRI11),fii,Fundo Imobiliário (FII),1.25,99.2,53.05,FII - Multicategoria,100.0,NaN
12,Kinea High Yield CRI FII (KNHY11),fii,Fundo Imobiliário (FII),1.60,98.8,35.23,FII - Multicategoria,100.0,NaN
9,Kinea Infra FII (KDIF11),fii,Renda Fixa,1.05,97.2,54.66,Renda Fixa Duração Livre Crédito Livre,50.0,"quantidade_cotistas, valor_patrimonial"
13,Kinea Andes FIF CIC RF CP LP RL,fundo_aberto,Renda Fixa,0.70,94.2,24.38,Renda Fixa Duração Livre Crédito Livre,100.0,NaN
3,Kinea Oportunidade FIF RF CP RL - Subclasse I,fundo_aberto,Renda Fixa,0.80,88.3,10.55,Renda Fixa Duração Livre Crédito Livre,100.0,NaN
10,Kinea Fundo de Fundos Imobiliarios FII (KFOF11),fii,Fundo Imobiliário (FII),0.92,88.1,7.13,FII - Multicategoria,100.0,NaN


## 9. Diagnóstico preliminar — bem posicionado / desvantagem / mal comunicado (17 fundos)

In [142]:
# Critérios objetivos, documentados aqui (não escondidos em número mágico):
# - Porte líder: percentil_pl >= 90 | Porte pequeno: percentil_pl <= 20
# - Ficha incompleta: completude_conteudo_pct < 100
# - Taxa fora do padrão Kinea: mais de 0.3 p.p. acima/abaixo da média das
#   OUTRAS Kinea da mesma categoria_padronizada (comparação interna - não
#   substitui comparação com concorrente externo, que fica pros 3 escolhidos)

LIMIAR_PORTE_LIDER = 90
LIMIAR_PORTE_PEQUENO = 20
LIMIAR_TAXA_DESVIO_PP = 0.3

df_visao = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "visao_consolidada.csv")

media_categoria = df_visao.groupby("categoria_padronizada")["taxa_administracao_pct"].transform("mean")
df_visao["taxa_vs_media_categoria_kinea"] = (df_visao["taxa_administracao_pct"] - media_categoria).round(2)

def diagnostico(row):
    achados = []
    if row["percentil_pl"] >= LIMIAR_PORTE_LIDER:
        achados.append("porte líder na categoria")
    elif row["percentil_pl"] <= LIMIAR_PORTE_PEQUENO:
        achados.append("porte pequeno na categoria")
    if row["completude_conteudo_pct"] < 100:
        achados.append(f"ficha incompleta ({row['completude_conteudo_pct']}%: falta {row['campos_faltando']})")
    if row["taxa_vs_media_categoria_kinea"] > LIMIAR_TAXA_DESVIO_PP:
        achados.append("taxa adm. acima da média das outras Kinea da mesma categoria")
    elif row["taxa_vs_media_categoria_kinea"] < -LIMIAR_TAXA_DESVIO_PP:
        achados.append("taxa adm. abaixo da média das outras Kinea da mesma categoria")
    return "; ".join(achados) if achados else "sem achado nos critérios acima"

df_visao["diagnostico_preliminar"] = df_visao.apply(diagnostico, axis=1)

OUT_DIAGNOSTICO = PROJECT_ROOT / "data" / "processed" / "diagnostico_preliminar.csv"
df_visao.to_csv(OUT_DIAGNOSTICO, index=False, encoding="utf-8")
print("Salvo em:", OUT_DIAGNOSTICO)
df_visao[["nome_padronizado", "categoria_padronizada", "percentil_pl", "taxa_vs_media_categoria_kinea", "completude_conteudo_pct", "diagnostico_preliminar"]]

Salvo em: /Users/julianamurakami/Downloads/case-kinea-bi/data/processed/diagnostico_preliminar.csv


,nome_padronizado,categoria_padronizada,percentil_pl,taxa_vs_media_categoria_kinea,completude_conteudo_pct,diagnostico_preliminar
0,Kinea Credito Agro FIAGRO FII (KNCA11),Fundo Imobiliário (FII),100.0,-0.13,50.0,porte líder na categoria; ficha incompleta (50...
1,Kinea Rendimentos Imobiliarios FII (KNCR11),Fundo Imobiliário (FII),100.0,-0.13,100.0,porte líder na categoria
2,Kinea Indices de Precos FII (KNIP11),Fundo Imobiliário (FII),99.7,-0.13,100.0,porte líder na categoria
3,Kinea Alpes Prev XP Seg RF CP FICFI,Multimercado,99.5,0.00,66.7,porte líder na categoria; ficha incompleta (66...
4,Kinea Renda Imobiliaria FII (KNRI11),Fundo Imobiliário (FII),99.2,0.12,100.0,porte líder na categoria
5,Kinea High Yield CRI FII (KNHY11),Fundo Imobiliário (FII),98.8,0.47,100.0,porte líder na categoria; taxa adm. acima da m...
6,Kinea Infra FII (KDIF11),Renda Fixa,97.2,0.25,50.0,porte líder na categoria; ficha incompleta (50...
7,Kinea Andes FIF CIC RF CP LP RL,Renda Fixa,94.2,-0.10,100.0,porte líder na categoria
8,Kinea Oportunidade FIF RF CP RL - Subclasse I,Renda Fixa,88.3,-0.00,100.0,sem achado nos critérios acima
9,Kinea Fundo de Fundos Imobiliarios FII (KFOF11),Fundo Imobiliário (FII),88.1,-0.21,100.0,sem achado nos critérios acima


In [143]:
# --- Verificação: robustez do universo FIAGRO (KNCA11) e categorização do KDIF11 ---
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

concorrentes = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "fundos_concorrentes.csv")
universo = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "universo_kinea_completo.csv")

# 1) Tamanho do universo competitivo do KNCA11 (classe FIAGRO)
fiagro = concorrentes[concorrentes['classe_anbima'] == 'FIAGRO']
n_fiagro_total = len(fiagro)
n_fiagro_concorrentes = len(fiagro[~fiagro['eh_fundo_kinea']])
print(f"Universo FIAGRO: {n_fiagro_total} fundos totais, {n_fiagro_concorrentes} concorrentes (excluindo Kinea)")

# 2) Checagem de consistência de categorização: KDIF11
kdif = universo[universo['nome_padronizado'].str.contains('Infra', case=False, na=False)]
print(kdif[['nome_padronizado', 'categoria_padronizada', 'classe_anbima_cvm']])

Universo FIAGRO: 8 fundos totais, 5 concorrentes (excluindo Kinea)
           nome_padronizado categoria_padronizada  \
9  Kinea Infra FII (KDIF11)            Renda Fixa   

                        classe_anbima_cvm  
9  Renda Fixa Duração Livre Crédito Livre  


In [144]:
from pathlib import Path

repo_root = Path("..").resolve()
print("repo_root:", repo_root)
print()

raw = repo_root / "data" / "raw"
if raw.exists():
    print("conteúdo de data/raw:")
    for f in raw.iterdir():
        print(" -", f.name)
else:
    print("data/raw não existe nem a partir da raiz. Procurando o zip em todo o repo...")
    for p in repo_root.rglob("*registro_fundo_classe*"):
        print(" encontrado em:", p)
    print()
    print("estrutura de data/ (se existir):")
    data_dir = repo_root / "data"
    if data_dir.exists():
        for p in data_dir.rglob("*"):
            if p.is_file():
                print(" -", p.relative_to(repo_root))

repo_root: /Users/julianamurakami/Downloads/case-kinea-bi

conteúdo de data/raw:
 - .DS_Store
 - universo_kinea_raw_manual_original.csv
 - cvm_cad_fi.csv
 - html_concorrentes
 - universo_kinea_raw.csv
 - cvm_registro_fundo_classe.zip.meta.txt
 - html
 - cvm_cad_fi.csv.meta.txt
 - cvm_registro_fundo_classe.zip
 - universo_kinea_raw_scraped.csv
 - concorrentes_aprofundamento_raw.csv
 - fetched_content


In [145]:
import pandas as pd
import zipfile
from pathlib import Path

repo_root = Path("..").resolve()
zip_path = repo_root / "data" / "raw" / "cvm_registro_fundo_classe.zip"

with zipfile.ZipFile(zip_path) as z:
    nomes = z.namelist()
    print("arquivos dentro do zip:")
    for n in nomes:
        print(" -", n)
    print()
    nome_classe = next(n for n in nomes if "classe" in n.lower() and "sub" not in n.lower())
    with z.open(nome_classe) as f:
        df_classe_preview = pd.read_csv(f, sep=";", encoding="latin-1", nrows=3, low_memory=False)

print("colunas de", nome_classe, ":")
print(list(df_classe_preview.columns))

arquivos dentro do zip:
 - registro_classe.csv
 - registro_fundo.csv
 - registro_subclasse.csv

colunas de registro_classe.csv :
['ID_Registro_Fundo', 'ID_Registro_Classe', 'CNPJ_Classe', 'Codigo_CVM', 'Data_Registro', 'Data_Constituicao', 'Data_Inicio', 'Tipo_Classe', 'Denominacao_Social', 'Situacao', 'Data_Inicio_Situacao', 'Classificacao', 'Indicador_Desempenho', 'Classe_Cotas', 'Classificacao_Anbima', 'Tributacao_Longo_Prazo', 'Entidade_Investimento', 'Permitido_Aplicacao_CemPorCento_Exterior', 'Classe_ESG', 'Forma_Condominio', 'Exclusivo', 'Publico_Alvo', 'Patrimonio_Liquido', 'Data_Patrimonio_Liquido', 'CNPJ_Auditor', 'Auditor', 'CNPJ_Custodiante', 'Custodiante', 'CNPJ_Controlador', 'Controlador']


In [146]:
import pandas as pd
df_check = pd.read_csv(repo_root / "data" / "raw" / "concorrentes_aprofundamento_raw.csv")
print(df_check.shape)
print(df_check.columns.tolist())
print(df_check.head(3))

(8, 73)
['nome_referencia', 'url', 'tipo_pagina', 'source', 'access_timestamp', 'extraction_method', 'http_status', 'titulo_pagina', 'erro', 'administrador', 'aplicacao_minima', 'auditor', 'benchmark', 'classificacao_cvm', 'classificacao_xp', 'cnpj', 'cotas_emitidas', 'cotizacao_aplicacao', 'cotizacao_resgate', 'custodiante', 'data_inicio', 'dividend_yield', 'gestor', 'liquidacao_resgate', 'movimentacao_minima', 'objetivo', 'participacao_ifix', 'politica_gestao', 'publico_alvo', 'quantidade_cotistas', 'rating_morningstar', 'rent_cdi_3m', 'rent_cdi_6m', 'rent_cdi_dia', 'rent_cdi_mes', 'rent_cdi_no_ano', 'rent_cdi_semana', 'rent_fundo_3m', 'rent_fundo_6m', 'rent_fundo_dia', 'rent_fundo_mes', 'rent_fundo_no_ano', 'rent_fundo_semana', 'rent_ibov_3m', 'rent_ibov_6m', 'rent_ibov_dia', 'rent_ibov_mes', 'rent_ibov_no_ano', 'rent_ibov_semana', 'risco_pontuacao_xp', 'rr_rentabilidade_12m', 'rr_rentabilidade_24m', 'rr_rentabilidade_36m', 'rr_rentabilidade_desde_inicio', 'rr_rentabilidade_no_ano',

In [147]:
print('publico_alvo preenchidos:', df_concorrentes['publico_alvo'].notna().sum(), 'de', len(df_concorrentes))
print(df_concorrentes['publico_alvo'].value_counts(dropna=False).head(10))

publico_alvo preenchidos: 23290 de 27938
Profissional     13550
Público Geral     6049
Qualificado       3691
None              3308
NaN               1340
Name: publico_alvo, dtype: int64


In [148]:
import src.analysis.build_competitive_universe as bcu
import inspect

print("Arquivo carregado:", bcu.__file__)
print()
print(inspect.getsource(bcu.carregar_registro_classe_fundo))

Arquivo carregado: /Users/julianamurakami/Downloads/case-kinea-bi/src/analysis/build_competitive_universe.py

def carregar_registro_classe_fundo(zip_path: Path) -> pd.DataFrame:
    """Fonte 1 (primária): fundos/classes adaptados à RCVM175."""
    with zipfile.ZipFile(zip_path) as z:
        nomes = z.namelist()
        nome_classe = next(n for n in nomes if "classe" in n.lower() and "sub" not in n.lower())
        nome_fundo = next(n for n in nomes if "fundo" in n.lower() and "classe" not in n.lower())
        with z.open(nome_classe) as f:
            df_classe = pd.read_csv(f, sep=";", encoding="latin-1", low_memory=False)
        with z.open(nome_fundo) as f:
            df_fundo = pd.read_csv(f, sep=";", encoding="latin-1", low_memory=False)
    df_classe.columns = [c.strip() for c in df_classe.columns]
    df_fundo.columns = [c.strip() for c in df_fundo.columns]

    df_classe_ativas = df_classe[
        df_classe["Situacao"].astype(str).str.contains("FUNCIONAMENTO NORMAL", case=

In [149]:
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df_concorrentes = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "fundos_concorrentes.csv")

taxa_fii = df_concorrentes[
    (df_concorrentes["classe_anbima"] == "FII - Multicategoria")
    & df_concorrentes["taxa_administracao_cvm_mensal_pct"].notna()
].copy()

print(f"Total com taxa preenchida: {len(taxa_fii)}")
print()

# Limite plausível: taxa de administração de FII raramente passa de ~0,25% ao
# mês (~3% a.a.) na prática de mercado. Negativo não existe. Usamos 0,25%
# como teto de sanidade (não é filtro definitivo, é pra ver o tamanho do problema).
LIMITE_MENSAL = 0.0025

negativos = taxa_fii[taxa_fii["taxa_administracao_cvm_mensal_pct"] < 0]
acima_limite = taxa_fii[taxa_fii["taxa_administracao_cvm_mensal_pct"] > LIMITE_MENSAL]

print(f"Negativos: {len(negativos)} ({len(negativos)/len(taxa_fii)*100:.1f}%)")
print(f"Acima de {LIMITE_MENSAL*100:.2f}%/mês: {len(acima_limite)} ({len(acima_limite)/len(taxa_fii)*100:.1f}%)")
print()

print("=== Amostra dos negativos ===")
print(negativos[["nome", "taxa_administracao_cvm_mensal_pct"]].head(10).to_string(index=False))
print()

print("=== Amostra dos acima do limite ===")
print(acima_limite[["nome", "taxa_administracao_cvm_mensal_pct"]].sort_values(
    "taxa_administracao_cvm_mensal_pct", ascending=False
).head(10).to_string(index=False))
print()

# Comparação: mediana com todos os dados vs. mediana só com valores plausíveis
taxa_sane = taxa_fii[
    (taxa_fii["taxa_administracao_cvm_mensal_pct"] >= 0)
    & (taxa_fii["taxa_administracao_cvm_mensal_pct"] <= LIMITE_MENSAL)
]
print(f"Mediana com TODOS os dados (n={len(taxa_fii)}): {taxa_fii['taxa_administracao_cvm_mensal_pct'].median():.6f}")
print(f"Mediana só com valores plausíveis (n={len(taxa_sane)}): {taxa_sane['taxa_administracao_cvm_mensal_pct'].median():.6f}")

Total com taxa preenchida: 3300

Negativos: 20 (0.6%)
Acima de 0.25%/mês: 315 (9.5%)

=== Amostra dos negativos ===
                                    nome  taxa_administracao_cvm_mensal_pct
       BLUECAP LAST MILE I FII RESP LTDA                          -0.097052
               KRONOLOG II FII RESP LTDA                          -0.000069
ETT BLUECAP FII RESPONSABILIDADE LIMITAD                          -0.161913
         FII PLANNER 3837 - SPB MALL FII                          -0.347497
       BLUECAP LAST MILE I FII RESP LTDA                          -0.097052
               KRONOLOG II FII RESP LTDA                          -0.000069
ETT BLUECAP FII RESPONSABILIDADE LIMITAD                          -0.161913
         FII PLANNER 3837 - SPB MALL FII                          -0.347497
       BLUECAP LAST MILE I FII RESP LTDA                          -0.097052
               KRONOLOG II FII RESP LTDA                          -0.000069

=== Amostra dos acima do limite ===
           

In [150]:
LIMITE_MENSAL = 0.0025  # mesmo teto de sanidade da célula anterior

def taxa_sane_mediana(df: pd.DataFrame) -> tuple:
    """Filtra taxa de administração pra faixa plausível e retorna (mediana, n_usado, n_excluido)."""
    validos = df["taxa_administracao_cvm_mensal_pct"].notna()
    dentro_faixa = (df["taxa_administracao_cvm_mensal_pct"] >= 0) & (df["taxa_administracao_cvm_mensal_pct"] <= LIMITE_MENSAL)
    df_sane = df[validos & dentro_faixa]
    n_excluido = df[validos].shape[0] - df_sane.shape[0]
    return df_sane["taxa_administracao_cvm_mensal_pct"].median(), len(df_sane), n_excluido

# Universo amplo do KNHY11 especificamente (não o combinado de todos os FII-Multicategoria)
concorrentes_knhy11_amplo = df_concorrentes[
    (df_concorrentes["referencia_fundo_kinea"] == "Kinea High Yield CRI FII (KNHY11)")
    & (~df_concorrentes["eh_fundo_kinea"])
]

mediana_mensal, n_usado, n_excluido = taxa_sane_mediana(concorrentes_knhy11_amplo)
mediana_anual_pct = round(mediana_mensal * 1200, 2)

print(f"Universo amplo KNHY11: {len(concorrentes_knhy11_amplo)} concorrentes totais")
print(f"Com taxa válida (dentro da faixa 0-{LIMITE_MENSAL*100:.2f}%/mês): {n_usado}")
print(f"Excluídos por implausibilidade (negativo ou acima do teto): {n_excluido}")
print(f"Mediana anualizada estimada: {mediana_anual_pct}%")

Universo amplo KNHY11: 647 concorrentes totais
Com taxa válida (dentro da faixa 0-0.25%/mês): 580
Excluídos por implausibilidade (negativo ou acima do teto): 67
Mediana anualizada estimada: 0.43%


In [151]:
concorrentes_knca11 = df_concorrentes[
    (df_concorrentes["referencia_fundo_kinea"].str.contains("KNCA11", case=False, na=False))
    & (~df_concorrentes["eh_fundo_kinea"])
]
print(f"{len(concorrentes_knca11)} concorrentes do KNCA11:")
print(concorrentes_knca11[["nome", "cnpj", "classe_anbima", "patrimonio_liquido"]].to_string(index=False))

5 concorrentes do KNCA11:
                                                      nome               cnpj classe_anbima  patrimonio_liquido
         ITAÚ ASSET RURAL FIAGRO RESPONSABILIDADE LIMITADA 42.479.593/0001-60        FIAGRO        1.646198e+09
                VECTIS DATAGRO CRÉDITO AGRONEGÓCIO FI LTDA 42.502.827/0001-43        FIAGRO        4.659891e+08
  CANAÃ - FIAGRO - IMOBILIÁRIO - RESPONSABILIDADE LIMITADA 45.123.558/0001-00        FIAGRO        1.548232e+09
KOPPERT FUNDO INVEST. NAS CADEIAS PROD. DO AGRONEGÓCIO  RL 47.669.421/0001-73        FIAGRO        1.987062e+08
  HEDGE I FIAGRO - PARTICIP - MULTIESTRATÉGIA DE RESP LTDA 49.371.794/0001-99        FIAGRO        2.721545e+06


In [152]:
import importlib
import src.analysis.build_competitive_universe as bcu
importlib.reload(bcu)

df_kinea = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "universo_kinea.csv")
df_resumo_novo, df_concorrentes_novo = bcu.montar_universo_competitivo(df_kinea, PROJECT_ROOT)

print("Colunas do df_resumo_novo:", df_resumo_novo.columns.tolist())
print()
print(df_resumo_novo[["fundo_kinea", "n_universo_inicial", "n_concorrentes_diretos"]])

# Salva por cima dos CSVs oficiais
df_resumo_novo.to_csv(PROJECT_ROOT / "data" / "processed" / "universo_competitivo_resumo.csv", index=False)
df_concorrentes_novo.to_csv(PROJECT_ROOT / "data" / "processed" / "fundos_concorrentes.csv", index=False)
print("\nSalvo.")

Colunas do df_resumo_novo: ['fundo_kinea', 'categoria_xp', 'classe_anbima_cvm', 'n_universo_inicial', 'n_concorrentes_diretos', 'observacao']

                                        fundo_kinea  n_universo_inicial  \
0                Kinea Chronos FIM RL - Subclasse I                 648   
1       Kinea Rendimentos Imobiliarios FII (KNCR11)                 660   
2               Kinea Atlas II FIM RL - Subclasse I                 648   
3     Kinea Oportunidade FIF RF CP RL - Subclasse I                3062   
4      Kinea Gama FIF CIC em Acoes RL - Subclasse I                1856   
5   Kinea IPCA Dinamico II FIF RF RL - Subclasse II                3062   
6               Kinea Alpes Prev XP Seg RF CP FICFI                2458   
7              Kinea Renda Imobiliaria FII (KNRI11)                 660   
8              Kinea Indices de Precos FII (KNIP11)                 660   
9                          Kinea Infra FII (KDIF11)                3062   
10  Kinea Fundo de Fundos Imobil